# Stem cell model for solving network-related problems
Example problem: MTU (maximum transmission unit) on OSI layer 3 was set to 1400 bytes. That prohibits OSPF (a routing protocol) from reaching Full state. Because of that, layer 7 services (RabbitMQ and gRPC) are unable to exchange data. The model receives logs from:
- Cisco IOS (router setup)
- OSPF
- RabbitMQ
- gRPC

and has to transform itself from a generic model to a networking specialist, able to find causal connection between settings set on layer 3 and problems detected at layer 7.

In [1]:
import model
settings = model.Settings()
basic_agent = model.Model("gpt-4o", settings.openai_api_key)

## 1. Collecting resources to a single string

In [2]:
context_string = basic_agent.generate_context_string("resources")
print(context_string)

Source cisco-ios-console:
Core-Router-A# show running-config interface GigabitEthernet0/1
Building configuration...

Current configuration : 214 bytes
!
interface GigabitEthernet0/1
 description CONNECTION_TO_DATACENTER_CORE
 mac-address 00ab.cd12.3456
 ip address 10.254.1.1 255.255.255.252
 ip mtu 1400
 ip ospf message-digest-key 1 md5 7 0822455D0A16
 ip ospf dead-interval 40
 ip ospf hello-interval 10
 load-interval 30
 negotiation auto
end

Source gRPC-log:
[2026-05-13T17:48:10.102Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExceeded desc = context deadline exceeded
[2026-05-13T17:48:10.105Z] [WARN] [BillingSvc] Payload size (bytes): 658402, attempt: 1/5, initiating retry in 500ms...
[2026-05-13T17:48:10.608Z] [INFO] [BillingSvc] Retrying InvoiceProcessor.SubmitBatch (correlation_id: 8f9a2b1c-4d5e-6f7a)
[2026-05-13T17:48:15.612Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExc

## 2. First cell differentiation (first API call)

In [3]:
model_results = basic_agent.differentiate(context_string, 5)
print(model_results)

SPECIALIZATION_NAME:
Network Performance Analyst
----------------------------------------
REASONING_SUMMARY:
The environment telemetry indicates a network performance issue affecting gRPC calls, likely due to MTU size mismatch or congestion. The gRPC logs show repeated deadline exceeded errors, suggesting packet loss or delays. The router configuration shows an MTU of 1400, which might be causing fragmentation if the payload size exceeds this limit. Additionally, the high message count in the RabbitMQ queue suggests potential congestion or processing delays.
----------------------------------------
ROUTE_CAUSE_HYPOTHESIS:
The gRPC deadline exceeded errors are likely caused by packet fragmentation due to the MTU size of 1400 on the GigabitEthernet0/1 interface, which is lower than the standard 1500 bytes. This could lead to increased latency and packet loss, especially for large payloads like the 658402 bytes seen in the logs. Additionally, the high message count in the RabbitMQ billing

It seems that the model correctly identified the problem and formed a correct hypothesis.